<a href="https://colab.research.google.com/github/Ayush-0108/FinBee/blob/main/layer2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 6.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.9 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=0e9e8b749a798a3575c0bb7dde1e58a0df378ebd67dc793d9430b0f7b73b7c42
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [2]:
!pip install torch torchvision torchaudio

In [3]:
!pip install pydub streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 49.1 MB/s eta 0:00:00


In [4]:
%%time
import whisper

CPU times: user 3.87 s, sys: 482 ms, total: 4.35 s
Wall time: 8.9 s


In [53]:
%%time
whisper_model =  whisper.load_model("base")

CPU times: user 1.64 s, sys: 1.16 s, total: 2.81 s
Wall time: 3.67 s


In [35]:
%%time
from IPython.display import Audio
Audio("/content/lic_question.ogg")

CPU times: user 1.47 ms, sys: 92 µs, total: 1.56 ms
Wall time: 1.35 ms


In [54]:
%%time
import torch

audio = whisper.load_audio("/content/loan_emiquestion.ogg")
audio = whisper.pad_or_trim(audio)

# Explicitly determine the device
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

mel = whisper.log_mel_spectrogram(audio).to(device)

_,probs = whisper_model.detect_language(mel)
print(f"Detected language: {max(probs, key=probs.get)}")

options = whisper.DecodingOptions()
result = whisper.decode(whisper_model, mel, options)

print(result.text)

Detected language: ur
میرے کو دانسے لون لینہ ہے جس کی ہے محنی میرے کو ہر محنی 10 سر بڑے گی اور میرے ہر محنی کی تنگوہ 20 سر پہ ہے تو کیا مجھے یہ لون لینہ چاہیے
CPU times: user 23.4 s, sys: 2.89 s, total: 26.3 s
Wall time: 28.5 s


In [8]:
!pip install git+https://github.com/speechbrain/speechbrain.git@develop


  Cloning https://github.com/speechbrain/speechbrain.git (to revision develop) to /tmp/pip-req-build-ih9r8y8q
  Running command git clone --filter=blob:none --quiet https://github.com/speechbrain/speechbrain.git /tmp/pip-req-build-ih9r8y8q
  Resolved https://github.com/speechbrain/speechbrain.git to commit aca5e41a34b3bcbe532640f2d3d0f6de3fd50ebe
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 10.3 MB/s eta 0:00:00
  Created wheel for speechbrain: filename=speechbrain-1.0.3-py3-none-any.whl size=2277702 sha256=e3f3fe9376c317ac12ad1a0e741fec8f4baffd98d87f15bcca9693fb63f9c910
  Stored in directory: /tmp/pip-ephem-wheel-cache-ih3b0574/wheels/dc/ef/85/c59a2253075875b12cfead53afa99da95b99a83aa189f9bfa6
Successfully built speechbrain


In [11]:
%%time
import torchaudio
from speechbrain.inference.classifiers import EncoderClassifier
language_id = EncoderClassifier.from_hparams(source="speechbrain/lang-id-voxlingua107-ecapa", savedir="tmp")
# Download Thai language sample from Omniglot and cvert to suitable form
signal = language_id.load_audio(r"/content/lic_question.ogg")
prediction =  language_id.classify_batch(signal)


# The scores in the prediction[0] tensor can be interpreted as log-likelihoods that
# the given utterance belongs to the given language (i.e., the larger the better)
# The linear-scale likelihood can be retrieved using the following:
print(prediction[1].exp())

# The identified language ISO code is given in prediction[3]
print(prediction[3])


INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/tmp/hyperparams.yaml'
DEBUG:speechbrain.utils.parameter_transfer:Collecting files (or symlinks) for pretraining in tmp.
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/tmp/embedding_model.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["embedding_model"] = /content/tmp/embedding_model.ckpt
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/tmp/classifier.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["classifier"] = /content/tmp/classifier.ckpt
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/tmp/label_encoder.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["label_encoder"] = /content/tmp/label_encoder.ckpt
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_

tensor([0.8657])
['hi: Hindi']
CPU times: user 2.05 s, sys: 766 ms, total: 2.81 s
Wall time: 3.71 s


In [12]:
raw_output = prediction[3][0]  # Get the first string: 'hi: Hindi'

# Split by the colon and take the first part, then remove any extra spaces
lang_id = raw_output.split(':')[0].strip()

print(f"Extracted ID for NeMo: {lang_id}")

Extracted ID for NeMo: hi


In [13]:
!pip install transformers torchaudio "onnxruntime==1.20.1" "onnx==1.20.1" "onnxruntime-gpu==1.20.1" -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.5/291.5 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 5.7 MB/s eta 0:00:00


In [46]:
%%time
from transformers import AutoModel
import torch, torchaudio

# Load the model
model = AutoModel.from_pretrained("ai4bharat/indic-conformer-600m-multilingual", trust_remote_code=True)

# Load an audio file
wav, sr = torchaudio.load("/content/lic_question.ogg")
wav = torch.mean(wav, dim=0, keepdim=True)

target_sample_rate = 16000  # Expected sample rate
if sr != target_sample_rate:
    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sample_rate)
    wav = resampler(wav)

# Perform ASR with CTC decoding
transcription_ctc = model(wav, lang_id, "ctc")
print("CTC Transcription:", transcription_ctc)

# # Perform ASR with RNNT decoding
# transcription_rnnt = model(wav, lang_id, "rnnt")
# print("RNNT Transcription:", transcription_rnnt)

Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate


Fetching 404 files:   0%|          | 0/404 [00:00<?, ?it/s]

Please check FRAME_DURATION_MS. The timestamps can be inaccurate
CTC Transcription: मैंने एल आई सी करवाया था हर हर साल एक लाख का जिसमें से मैंने सिर्फ दो साल भरा बट अब पाँच साल तक मैं भर नहीं पाया ड्यू टू कोविड ना वट टू वट आई हैव ड
CPU times: user 10.8 s, sys: 1.4 s, total: 12.2 s
Wall time: 24.6 s


In [15]:
# Install the helper for the optimized CPU version
!pip install indic-asr-onnx


In [16]:
%%time
from indic_asr_onnx import IndicTranscriber

# This automatically downloads the 8-bit quantized CPU-friendly model (~600MB)
transcriber = IndicTranscriber()

# Transcribe - 'hi' for Hindi, 'kn' for Kannada, etc.
# This will be much faster on your CPU than the AutoModel code.
text = transcriber.transcribe_ctc("/content/lic_question.ogg", lang_id)
print("Transcription:", text)

Fetching 32 files:   0%|          | 0/32 [00:00<?, ?it/s]

Using CPU
Transcription: मैंने एल आई सी का करवाया था ह हर साल एक लाख काक जिसमें से मैं दो साल भराहा बट अब पाँच साल तक मैं भर नहीं पाया ड्यू टू कोविड ना वट वट आ
CPU times: user 17.8 s, sys: 3.17 s, total: 21 s
Wall time: 32.9 s


In [17]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 1.9 MB/s eta 0:00:00


In [47]:
from groq import Groq
import os
from google.colab import userdata

def correct_asr_output(raw_text: str, language: str = "Hindi", context: str = "") -> str:
    # Fetch Groq API key from Colab secrets
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    client = Groq(api_key=GROQ_API_KEY)

    prompt = f"""You are a linguistic expert specializing in correcting Automatic Speech Recognition (ASR) output for multilingual speakers.

Language: {language}
Domain: {context if context else "financial/banking"}

Your task is to fix the following ASR errors WITHOUT changing the meaning:

1. **Grammar** – Fix verb forms and sentence structure.
2. **Homophones** – Replace words that sound similar but are wrong in context.
3. **Phonetic Restoration** – Restore full words (e.g., "tanwah" → "tankhwah").
4. **Acronyms** – Standardize financial terms (e.g., "em" → "EMI", "lic" → "LIC").
5. **ASR Artifacts** – Remove stutters ("the the") and fillers ("uh", "um").
6. **Numbers** – Standardize spoken numbers (e.g., "bees hazar" → "20,000").
7. **Punctuation** – Add necessary commas/periods for readability.
8. **Script Handling (CRITICAL)**: Detect English words written in {language} script or phonetically. Convert these specific words to standard English script (e.g., "लोन" → "Loan", "एमी" → "EMI"). Keep native {language} words in their original script.

Rules:
- DO NOT paraphrase or add new information.
- DO NOT remove words unless they are filler/stutter artifacts.
- Return ONLY the corrected text. No greetings, no explanations.

ASR Output: {raw_text}"""

    chat_completion = client.chat.completions.create(
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        model="llama-3.3-70b-versatile",
        max_tokens=1024,
        temperature=0.0 # Keep at 0 for consistency
    )

    return chat_completion.choices[0].message.content.strip()

    return chat_completion.choices[0].message.content.strip()

In [29]:
%%time
correct_asr_output(
    transcription_ctc,
    language=lang_id
)

CPU times: user 86.5 ms, sys: 2.49 ms, total: 89 ms
Wall time: 1.38 s


'मैंने LIC करवाया था हर साल एक लाख का, जिसमें से मैंने सिर्फ दो साल भरा था, लेकिन अब पाँच साल तक मैं भर नहीं पाया कोविड के कारण, ना वट टू वट, आई हेव ड्यू टू कोविड।'

In [43]:
vad_model, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                              model='silero_vad',
                              force_reload=False)

(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


In [44]:
def apply_vad(input_audio_path):
    """
    Removes silence and background noise, leaving only speech.
    Returns the path to the 'tightly cropped' audio file.
    """
    SAMPLING_RATE = 16000 # Critical for Indic-Conformer

    # Load audio using Silero's built-in reader
    wav = read_audio(input_audio_path, sampling_rate=SAMPLING_RATE)

    # Get speech timestamps (threshold=0.5 is standard)
    # This identifies the start/end of every spoken sentence
    speech_timestamps = get_speech_timestamps(wav, vad_model, sampling_rate=SAMPLING_RATE, threshold=0.5)

    # Merge all 'speech' chunks into one continuous audio signal
    if speech_timestamps:
        # collect_chunks handles the concatenation of voice segments
        wav_speech = collect_chunks(speech_timestamps, wav)

        output_path = "vad_cleaned_audio.wav"
        save_audio(output_path, wav_speech, sampling_rate=SAMPLING_RATE)
        return output_path
    else:
        # If no speech is detected at all
        return None

### First log of succesful ASR working of two distinct ASRs
* WHisper ASR
* Indic ASR by AI4Bharat

CPU times: user 4 µs, sys: 1 µs, total: 5 µs
Wall time: 9.54 µs
Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Please check FRAME_DURATION_MS. The timestamps can be inaccurate
Download complete:  0.00/0.00 [00:00<?, ?B/s]Fetching 404 files: 100% 404/404 [00:00<00:00, 4484.50it/s]Please check FRAME_DURATION_MS. The timestamps can be inaccurate
CTC Transcription: मेरे को बैंक से लोन लेना है जिसकी ई एम आई मेरे को हर महीने दस हजार पड़ेगी और मेरी हर महीने की तनख्वाह बीस हजार रुपये है तो क्या मुझे यह लोन लेना चाहिए
RNNT Transcription: मेरे को बैंक से लोन लेना है जिसकी ई एम आई मेरे को हर महीने दस हज़ार पड़ेगी और मेरी हर महीने की तनख्वाह बीस हजार रुपये है तो क्या मुझे यह लोन लेना चाहिए

**this the output of indic asr**

**-----------------------------------------**

Detected language: ur
میرے کو دانسے لون لینہ ہے جس کی ہے محنی میرے کو ہر محنی 10 سر بڑے گی اور میرے ہر محنی کی تنگوہ 20 سر پہ ہے تو کیا مجھے یہ لون لینہ چاہیے

**this is the output for whisper asr**

**-----------------------------------------**

**in indic asr total computation time is 40 secs thats the problem
whisper has 34 sec
which ever model we choose we have to reduce this time**

In [30]:
!pip install gradio -q

In [31]:
!pip install noisereduce librosa soundfile

In [52]:
import noisereduce as nr
import librosa
import soundfile as sf
import numpy as np

def finbee_clean_audio(input_path):
    # 1. Load audio (Librosa handles .ogg, .wav, .mp3)
    # We load at 16kHz immediately to save memory
    y, sr = librosa.load(input_path, sr=16000)

    # 2. Trim Silence (Removes the dead air at start/end)
    y_trimmed, _ = librosa.effects.trim(y, top_db=25)

    # 3. Noise Reduction (Spectral Gating)
    # This removes the constant "hiss" or "hum"
    y_clean = nr.reduce_noise(y=y_trimmed, sr=16000, prop_decrease=0.8)

    # 4. Normalization (Make sure the volume is consistent)
    y_final = librosa.util.normalize(y_clean)

    # 5. Save for the ASR model
    output_path = "cleaned_for_asr.wav"
    sf.write(output_path, y_final, 16000)

    return output_path
clean_file = finbee_clean_audio("/content/silence_test.ogg")
Audio(clean_file)
# TEST IT
# clean_file = finbee_clean_audio("lic_question.ogg")
# print(f"Cleaned file saved at: {clean_file}")

In [49]:
def transcribe(audio_file):
  voice_detection= apply_vad(audio_file)
  if voice_detection:
      output_path=finbee_clean_audio(voice_detection)
      language_id = EncoderClassifier.from_hparams(source="speechbrain/lang-id-voxlingua107-ecapa", savedir="tmp")
      signal = language_id.load_audio(output_path)
      prediction =  language_id.classify_batch(signal)
      lang_id= prediction[3][0].split(':')[0].strip()

      # Load an audio file
      wav, sr = torchaudio.load(audio_file)
      wav = torch.mean(wav, dim=0, keepdim=True)

      target_sample_rate = 16000  # Expected sample rate
      if sr != target_sample_rate:
          resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sample_rate)
          wav = resampler(wav)

      # Perform ASR with CTC decoding
      transcription_ctc = model(wav, lang_id, "ctc")
      return correct_asr_output(transcription_ctc,lang_id)
  else:
    return "No Audio Found"
transcribe("/content/2.ogg")


/usr/local/lib/python3.12/dist-packages/torchaudio/__init__.py:178: UserWarning: The 'bits_per_sample' parameter is not directly supported by TorchCodec AudioEncoder.
  return save_with_torchcodec(
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/tmp/hyperparams.yaml'
DEBUG:speechbrain.utils.parameter_transfer:Collecting files (or symlinks) for pretraining in tmp.
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/tmp/embedding_model.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["embedding_model"] = /content/tmp/embedding_model.ckpt
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/tmp/classifier.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["classifier"] = /content/tmp/classifier.ckpt
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/tmp/label_encoder.ckpt'
DEBUG:s

'हेलो, मेरा नाम आयुष है।'

In [34]:
import gradio as gr

In [27]:
gr.Interface(
    title = "FinBee",
    fn=transcribe,
    inputs=[gr.Audio(sources=['microphone','upload'], type="filepath")],
    outputs=[
        "textbox"
    ],
    live=True
).launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://decb9760b91db64293.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### This is the second log for the asr
* first things first I have chosen indic asr

workflow

vad->noise_filter->transcriber->llm

* need to work on noise filter(maybe try deepfilternet)